# Chunking examples (#23)

Examples and checks for `chunk_contract(text)` in `src/ingest/chunker.py`, which
implements the strategy decided in #22: 510 tokens per chunk with 128 tokens of overlap.

Each chunk comes back with its text and its `start_char` / `end_char` position in the
original contract, so `text[start_char:end_char]` is always exactly the chunk. #24 uses
those positions to work out which clauses fall in which chunk.

In [1]:
# Loading the data and the chunker
import sys
sys.path.insert(0, "..")          # so `src` can be imported from the notebooks folder

import json
import pandas as pd

from src.ingest.chunker import MAX_TOKENS, OVERLAP, chunk_contract, count_tokens

with open("../data/cuad/train_separate_questions.json") as f:
    train = json.load(f)

texts = {doc["title"]: doc["paragraphs"][0]["context"] for doc in train["data"]}

print("MAX_TOKENS:", MAX_TOKENS, "| OVERLAP:", OVERLAP)
print("train contracts:", len(texts))

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


MAX_TOKENS: 510 | OVERLAP: 128
train contracts: 408


In [2]:
# 1. A short contract: fits in one chunk
title = "NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT"
text = texts[title]
chunks = chunk_contract(text)

print(title)
print("characters:", len(text), "| tokens:", count_tokens(text), "| chunks:", len(chunks))
print()
pd.DataFrame(chunks).drop(columns="text")

NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT
characters: 1081 | tokens: 236 | chunks: 1



,chunk_index,start_char,end_char,n_tokens
0,0,0,1081,236


In [3]:
# The offsets give back exactly the chunk text.
c = chunks[0]
print("text[start_char:end_char] == chunk text:", text[c["start_char"]:c["end_char"]] == c["text"])
print()
print(c["text"][:400])

text[start_char:end_char] == chunk text: True

Exhibit 1

JOINT FILING AGREEMENT

The undersigned hereby agree to jointly prepare and file with the applicable regulatory authorities this Schedule 13G or Schedule 13D and any future amendments thereto reporting each of the undersigned's ownership of securities of the issuer named herein, and hereby affirm that such Schedule 13G or Schedule 13D is being filed on behalf of each of the undersigned 


In [4]:
# 2. The longest contract in train (by tokens)
title = max(texts, key=lambda t: count_tokens(texts[t]))
text = texts[title]
chunks = chunk_contract(text)
table = pd.DataFrame(chunks).drop(columns="text")

print(title)
print("characters:", len(text), "| words:", len(text.split()),
      "| tokens:", count_tokens(text), "| chunks:", len(chunks))
print()
print("first 3 chunks:")
print(table.head(3).to_string(index=False))
print()
print("last 2 chunks:")
print(table.tail(2).to_string(index=False))

MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-OUTSOURCING AGREEMENT


characters: 338211 | words: 41962 | tokens: 121051 | chunks: 317

first 3 chunks:
 chunk_index  start_char  end_char  n_tokens
           0           0      1338       510
           1         743      2763       510
           2        2382      3699       510

last 2 chunks:
 chunk_index  start_char  end_char  n_tokens
         315      336443    337808       509
         316      337454    338211       339


In [5]:
# Neighbouring chunks share 128 tokens of text, so a clause near a boundary is whole in one of them.
first, second = chunks[0], chunks[1]
shared = text[second["start_char"]:first["end_char"]]

print("chunk 0 ends at", first["end_char"], "| chunk 1 starts at", second["start_char"])
print("tokens shared by both:", count_tokens(shared))
print()
print("shared text:", " ".join(shared.split())[:300], "...")

chunk 0 ends at 1338 | chunk 1 starts at 743
tokens shared by both: 128

shared text: are hereby incorporated by reference ("Agreement") is entered into by and between International Business Machines Corporation, a corporation incorporated under the laws of New York, U.S.A., having an office for the transaction of business at 8501 IBM Drive, Charlotte, North Carolina 28262 ("IBM"), a ...


In [6]:
# 3. Check every train contract
n_chunks = n_bad_offsets = n_too_long = n_gaps = 0
n_spans = n_intact = 0

for doc in train["data"]:
    text = doc["paragraphs"][0]["context"]
    chunks = chunk_contract(text)
    n_chunks += len(chunks)

    for i, c in enumerate(chunks):
        if text[c["start_char"]:c["end_char"]] != c["text"]:
            n_bad_offsets += 1
        if c["n_tokens"] > MAX_TOKENS:
            n_too_long += 1
        # Anything between two chunks must be only whitespace.
        if i > 0 and text[chunks[i - 1]["end_char"]:c["start_char"]].strip():
            n_gaps += 1

    for qa in doc["paragraphs"][0]["qas"]:
        for a in qa["answers"]:
            start, end = a["answer_start"], a["answer_start"] + len(a["text"])
            n_spans += 1
            n_intact += any(c["start_char"] <= start and end <= c["end_char"] for c in chunks)

print("chunks:", n_chunks)
print("chunks whose offsets do not recover their text:", n_bad_offsets)
print("chunks over", MAX_TOKENS, "tokens:", n_too_long)
print("gaps between chunks:", n_gaps)
print(f"clauses kept whole: {n_intact} of {n_spans} ({100 * n_intact / n_spans:.1f}%)")

chunks: 14181
chunks whose offsets do not recover their text: 0
chunks over 510 tokens: 0
gaps between chunks: 0
clauses kept whole: 10931 of 11180 (97.8%)


In [7]:
# 4. Edge cases
print("empty text:      ", chunk_contract(""))
print("only whitespace: ", chunk_contract("   \n\n  "))
print("one word:        ", [(c["start_char"], c["end_char"], c["text"]) for c in chunk_contract("Hello")])

try:
    chunk_contract("some text", overlap=MAX_TOKENS)
except ValueError as e:
    print("overlap too big:  ValueError:", e)

# Tiny chunks make the overlap easy to see.
small = chunk_contract("The quick brown fox jumps over the lazy dog again and again.",
                       max_tokens=5, overlap=2)
print("5 tokens, overlap 2:", [c["text"] for c in small])

empty text:       []
only whitespace:  []
one word:         [(0, 5, 'Hello')]
overlap too big:  ValueError: overlap must be at least 0 and less than max_tokens, got 510
5 tokens, overlap 2: ['The quick brown fox jumps', 'fox jumps over the lazy', 'the lazy dog again and', 'again and again.']


## Notes

**Why chunks are sometimes a few tokens short of 510.** A window can start or end
partway through a word, and that piece can tokenize into more tokens on its own than it
did inside the full contract. On train, 1,750 of 14,181 windows (12%) came out 1–3
tokens over the limit when re-tokenized. `chunk_contract` trims those from the end until
they fit, so the model never silently cuts off the end of a chunk. The next chunk still
starts on schedule, so nothing is skipped, and clauses kept whole stay at 97.8%, the
same as #22 measured.

**What `n_tokens` means.** It counts tokens in the chunk text tokenized on its own, which
is how the model will see it. It does not include the 2 special tokens the model adds,
so every chunk is at most 512 tokens of model input.

**Speed.** Chunking all 408 train contracts takes under a minute, because every chunk is
re-tokenized to check its length. That's fine for one-off preprocessing.